In [7]:
gid_map = {
    'scalability': '1513474150',
    'optimality': '2117154112',
    'profiling_agg': '1122559114'
}

In [8]:
import gdown
import pandas as pd
import numpy as np
from scipy import stats

sheet_id = "1WUywPr_TxlUfPGVyMF0I2ALxXiHAyknKoZ3X13kKrUI"

def get_pandas_from_gid(gid_name):
    # Replace with your actual Google Sheet ID
    # Optionally: specify the sheet/tab number (gid=0 for the first sheet)
    gid = gid_map[gid_name]
    
    # Construct export URL
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
    
    # Set desired output filename
    output = f"{gid_name}.csv"
    
    # Download CSV using gdown
    gdown.download(url, output, quiet=False)
    original_df = pd.read_csv(output)
    return original_df

In [9]:
df = get_pandas_from_gid('profiling_agg')
df

/home/viniciusvdias/miniconda3/lib/python3.10/site-packages/gdown/parse_url.py:48: UserWarning: You specified a Google Drive link that is not the correct link to download a file. You might want to try `--fuzzy` option or the following url: https://drive.google.com/uc?id=None
  warnings.warn(
Downloading...
From: https://docs.google.com/spreadsheets/d/1WUywPr_TxlUfPGVyMF0I2ALxXiHAyknKoZ3X13kKrUI/export?format=csv&gid=1122559114
To: /home/viniciusvdias/repos/fractal-private/plots/profiling_agg.csv
1.15kB [00:00, 588kB/s]


,obj_function,graph,AVERAGE of obj_function_perc,STDEV of obj_function_perc,COUNT of obj_function_perc,confidence_interval
0,conductance,amazon,22.85,13.86,30,5.18
1,conductance,citeseer,21.14,12.96,30,4.84
2,conductance,dblp,4.75,0.44,30,0.16
3,conductance,patents,8.57,1.82,30,0.68
4,conductance,youtube,7.89,6.06,30,2.26
5,degreeentropy,amazon,68.24,12.48,30,4.66
6,degreeentropy,citeseer,61.05,11.69,30,4.37
7,degreeentropy,dblp,54.07,6.10,30,2.28
8,degreeentropy,patents,43.31,24.53,30,9.16
9,degreeentropy,youtube,63.80,5.33,30,1.99


In [11]:
df['obj_function'] = df['obj_function'].replace({
    'conductance': 'Conductance',
    'triangledensestsubgraph': 'Triangle Densest Subgraph',
    'densesubgraph': 'Densest Subgraph',
    'degreeentropy': 'Degree Entropy',
    'labelentropy': 'Label Entropy'
})

df['graph'] = df['graph'].replace({
    'citeseer': 'Citeseer',
    'amazon': 'Amazon',
    'dblp': 'DBLP',
    'patents': 'Patents',
    'livejournal': 'LiveJournal',
    'youtube': 'Youtube',
})

new_names = {
    'obj_function': 'Scoring Function',
    'graph': 'Graph',
    'AVERAGE of obj_function_perc': 'Time in Scoring Function (%)',
    'confidence_interval': 'Confidence Interval'
}

df_latex = df[list(new_names.keys())].rename(columns=new_names).copy()

df_latex['ci_symbol'] = "$\\pm$"
df_latex['Time in Scoring Function (%)'] = df_latex['Time in Scoring Function (%)'].astype(str) + df_latex['ci_symbol'] + df_latex['Confidence Interval'].astype(str)
df_latex = df_latex.drop(columns=['ci_symbol', 'Confidence Interval'])

graph_order = ['Citeseer', 'Amazon', 'DBLP', 'Patents', 'LiveJournal', 'Youtube']
scoring_function_order = ['Densest Subgraph', 'Triangle Densest Subgraph', 'Conductance', 'Degree Entropy', 'Label Entropy']

df_latex['Graph'] = pd.Categorical(df_latex['Graph'], categories=graph_order, ordered=True)
df_latex['Scoring Function'] = pd.Categorical(df_latex['Scoring Function'], categories=scoring_function_order, ordered=True)

df_latex = df_latex.sort_values(by=['Scoring Function', 'Graph'])

table_str = df_latex.to_latex(caption='Profiling of Scoring Functions', label='tab:scoring-functions', float_format="%.2f", multirow=True, index=False)
print(table_str)

\begin{table}
\caption{Profiling of Scoring Functions}
\label{tab:scoring-functions}
\begin{tabular}{lll}
\toprule
Scoring Function & Graph & Time in Scoring Function (%) \\
\midrule
Densest Subgraph & Citeseer & 0.58$\pm$0.09 \\
Densest Subgraph & Amazon & 0.54$\pm$0.09 \\
Densest Subgraph & DBLP & 0.32$\pm$0.07 \\
Densest Subgraph & Patents & 0.45$\pm$0.05 \\
Densest Subgraph & Youtube & 0.34$\pm$0.05 \\
Triangle Densest Subgraph & Citeseer & 71.46$\pm$2.0 \\
Triangle Densest Subgraph & Amazon & 77.34$\pm$1.82 \\
Triangle Densest Subgraph & DBLP & 85.75$\pm$1.13 \\
Triangle Densest Subgraph & Patents & 89.12$\pm$1.49 \\
Triangle Densest Subgraph & Youtube & 90.53$\pm$1.6 \\
Conductance & Citeseer & 21.14$\pm$4.84 \\
Conductance & Amazon & 22.85$\pm$5.18 \\
Conductance & DBLP & 4.75$\pm$0.16 \\
Conductance & Patents & 8.57$\pm$0.68 \\
Conductance & Youtube & 7.89$\pm$2.26 \\
Degree Entropy & Citeseer & 61.05$\pm$4.37 \\
Degree Entropy & Amazon & 68.24$\pm$4.66 \\
Degree Entropy & DBLP